# 4-1. RAG 기반 AI 에이전트 — 이론과 실습

> **📌 이 노트북에 대하여**
>
> 이번 챕터부터는 **별도의 PPT 교재 없이 이 노트북 하나로** 이론과 실습을 함께 진행한다.
> 설명을 읽고 -> 코드를 실행하고 -> 결과를 관찰하는 순서로 따라오면 된다.
>
> 복습할 때도 이 노트북이 곧 교재이므로, **설명 부분을 건너뛰지 말고 읽어보자.**

---

## 학습 목표

이 노트북을 마치면 다음을 할 수 있다.

1. LLM의 **환각(Hallucination)** 이 왜 발생하는지 설명하고, 직접 재현할 수 있다
2. **RAG의 동작 원리**를 준비 단계와 실행 단계로 나누어 설명할 수 있다
3. **키워드 검색과 의미 기반 검색**의 차이를 코드로 확인하고 설명할 수 있다
4. **임베딩·Vector Store·청킹**의 역할과 상호 관계를 이해한다
5. **LangGraph**로 RAG 파이프라인을 직접 구성할 수 있다

---

## 목차

| # | 내용 | 성격 |
|:---:|------|:---:|
| 0 | **환경 설정** — 패키지 설치, API 키, 모델 초기화 | 실습 |
| 1 | **LLM의 한계** — 환각(Hallucination) 현상 직접 체험 | 이론+실습 |
| 2 | **RAG란 무엇인가** — 개념, 아키텍처, 구성요소, 실무 사례 | 이론+실습 |
| 3 | **검색의 진화** — 키워드 검색 → 의미 기반 검색 | 이론+실습 |
| 4 | **임베딩과 Vector Store** — 텍스트를 숫자로, 숫자를 DB로 | 이론+실습 |
| 5 | **청킹 전략** — 문서를 어떻게 잘게 나눌 것인가 | 이론+실습 |
| 6 | **LangChain과 LangGraph** — 생태계 이해 + RAG 파이프라인 구축 | 이론+실습 |

---

## 선행 지식 — 앞 챕터와의 연결

이번 챕터는 앞에서 배운 내용 위에 쌓아 올린다.

| 앞 챕터 | 이번 챕터에서 어떻게 쓰이는가 |
|---|---|
| **2-1** 토큰화와 임베딩 | 임베딩 개념이 **문서 검색**의 핵심 도구가 된다 |
| **2-2** 합성 데이터 | GMS API 호출, 프롬프트 설계, 구조화 출력을 그대로 사용한다 |
| **3-1** 전이학습 | "이미 학습된 것을 활용한다"는 발상이 사전학습 LLM 활용으로 이어진다 |

<br>

> **💡 이번 챕터의 한 줄 요약**
>
> **"LLM은 똑똑하지만 우리 회사 문서는 모른다. 그럼 읽을 자료를 쥐여주자."**


---

## 0. 환경 설정

이번 실습에서는 **SSAFY GMS**를 통해 LLM을 호출한다.
2-2 챕터(합성 데이터)에서 사용한 것과 **완전히 동일한 방식**이다.

### API 키 준비

1. [GMS 사이트](https://gms.ssafy.io/web/)에 접속하여 본인의 **API Key를 복사**한다.
2. 이 노트북과 **같은 폴더**에 `.env` 파일을 만들고 아래 내용을 작성한다.

```
GMS_KEY="여기에_복사한_API키를_붙여넣기"
```

> **💡 왜 `.env` 파일을 쓰는가?**
>
> API 키를 코드에 직접 쓰면 GitHub 등에 그대로 올라가 **유출 사고**가 발생한다.
> `.env` 파일에 분리해 두고 `.gitignore`에 등록하면 커밋 대상에서 아예 제외된다.
> (2-2 챕터에서 다룬 내용과 동일하다)

> **⚠️ 자주 발생하는 오류**
>
> | 증상 | 원인 | 해결 |
> |---|---|---|
> | `GMS_KEY가 설정되지 않았습니다` | `.env`가 다른 폴더에 있음 | 노트북과 같은 폴더로 이동 |
> | 같은 증상 | 파일명이 `.env.txt` | 확장자 표시를 켜고 이름 수정 |
> | 같은 증상 | `.env` 수정 후 그대로 실행 | **커널 재시작** 후 재실행 |


In [ ]:
# 패키지 설치 (Docker 환경이라면 실행하지 않습니다.)
%pip install langchain langchain-openai langchain-community langgraph langchain-text-splitters langchain-pymupdf4llm chromadb tiktoken pymupdf python-dotenv

In [ ]:
import warnings
from os import getenv
from pathlib import Path       # 파일 경로는 pathlib으로 다룬다
from dotenv import load_dotenv
from pprint import pprint
warnings.filterwarnings('ignore')

# .env 파일에 적힌 키=값 쌍을 운영체제의 환경 변수로 등록한다.
# 이후 getenv()로 해당 값을 파이썬 코드 안에서 꺼내 쓸 수 있다.
load_dotenv()

# 등록된 환경 변수에서 GMS API 키를 가져온다
GMS_KEY = getenv('GMS_KEY')

if GMS_KEY:
    print('API 키 로드 성공!')
else:
    print('ERROR: .env 파일에 GMS_KEY가 설정되지 않았습니다.')
    print('이 노트북과 같은 폴더에 .env 파일을 생성하고 API 키를 입력하세요.')

print('환경 설정 완료')

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ═══════════════════════════════════════════════════════════
# SSAFY GMS 설정
# ═══════════════════════════════════════════════════════════
# GMS는 OpenAI API를 중계(proxy)하는 서비스이므로,
# 평소 쓰던 langchain_openai를 그대로 사용하고 주소(base_url)만 바꿔주면 된다.
GMS_BASE_URL = 'https://gms.ssafy.io/gmsapi/api.openai.com/v1/'

# ── LLM ──
# use_responses_api=True : 구버전 Chat Completions 대신 최신 Responses API를 사용
# reasoning_effort       : GPT-5 계열의 '생각하는 양'을 조절 (low / medium / high)
#                          예전의 temperature 자리를 대신하는 파라미터다.
# ⚠️ temperature는 지정하지 않는다.
#    GPT-5 계열 추론 모델은 기본값(1.0)만 허용하며,
#    다른 값을 넣으면 'Unsupported parameter' 오류가 발생한다.
llm = ChatOpenAI(
    model='gpt-5-nano',
    api_key=GMS_KEY,
    base_url=GMS_BASE_URL,
    use_responses_api=True,
    reasoning_effort='low',   # RAG 답변 생성은 자료를 정리하는 작업이라 low로 충분
)

# ── 임베딩 모델 ──
# 텍스트를 1536차원 벡터로 변환한다. (챕터 4에서 자세히 다룬다)
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=GMS_KEY,
    base_url=GMS_BASE_URL,
)

print('모델 초기화 완료')
print(f'  LLM       : {llm.model_name} (Responses API, effort={llm.reasoning_effort})')
print(f'  Embedding : {embeddings.model}')

# ========== 연결 확인 ==========
# 본격적인 실습 전에 GMS 연결이 정상인지 먼저 점검한다.
try:
    test = llm.invoke('안녕? 한 문장으로 자기소개 해줘.')
    pprint(f'\n✅ LLM 연결 정상 | 응답: {test.content[1]['text'][:100]}')
except Exception as e:
    print(f'\n❌ LLM 연결 실패: {type(e).__name__}')
    pprint(f'   {str(e)[:200]}')
    print('   -> .env의 GMS_KEY, 그리고 모델명을 확인하세요.')

try:
    v = embeddings.embed_query('연결 테스트')
    print(f'✅ 임베딩 연결 정상 | 벡터 차원: {len(v)}')
except Exception as e:
    print(f'❌ 임베딩 연결 실패: {type(e).__name__}')
    pprint(f'   {str(e)[:200]}')
    print('   -> 아래 [대안] 셀의 로컬 임베딩을 사용하세요.')

> **📌 Responses API란?**
>
> OpenAI는 두 가지 방식의 API를 제공한다.
>
> | 구분 | Chat Completions (기존) | **Responses (권장)** |
> |---|---|---|
> | 위상 | 유지보수 모드 | **최신 기능이 추가되는 쪽** |
> | 추론 제어 | 제한적 | `reasoning_effort` 지원 |
> | LangChain 설정 | 기본값 | `use_responses_api=True` |
>
> 2-2 챕터에서 배운 내용과 동일하다.
> LangChain을 쓰면 옵션 하나만 켜면 되므로, 호출 코드(`llm.invoke(...)`)는 그대로다.

> **💡 `reasoning_effort` — temperature를 대신하는 손잡이**
>
> | effort | 특징 | 적합한 작업 |
> |---|---|---|
> | `low` | 빠르고 저렴 | 자료 정리, 요약, 단순 판정 ← **이번 실습** |
> | `medium` | 균형 (기본값) | 일반적인 생성 |
> | `high` | 느리고 비쌈 | 복잡한 추론, 수학 |
>
> RAG의 답변 생성은 **검색된 자료를 정리해서 전달하는 작업**이라 `low`로 충분하다.
> 답변이 너무 단순하게 느껴진다면 `medium`으로 올려서 비교해 보자.
>
> ⚠️ 눈에 보이지 않는 **'생각하는 토큰'도 과금 대상**이므로, 작업 난이도에 맞춰 고르는 것이 중요하다.

> **⚠️ 임베딩 연결이 실패했다면 — 로컬 모델로 대체할 수 있다**
>
> 교육 계정의 권한 설정에 따라 임베딩 모델(`text-embedding-3-small`)을
> 사용하지 못할 수 있다. 이 경우 아래 셀을 실행해 **로컬 임베딩 모델**로 교체하면
> 이후 실습을 그대로 진행할 수 있다.
>
> | 구분 | GMS 임베딩 | 로컬 임베딩 |
> |---|---|---|
> | 방식 | API 호출 | **내 컴퓨터에서 직접 계산** |
> | 비용 | 토큰당 과금 | **무료** |
> | 차원 | 1536 | 768 |
> | 준비 | 없음 | 최초 1회 모델 다운로드(약 400MB) |
>
> 💡 **임베딩 연결이 정상이라면 아래 셀은 실행하지 않아도 된다.**


In [ ]:
# ========== [대안] 로컬 임베딩 모델 사용 ==========
# ⚠️ 위 셀에서 "임베딩 연결 실패"가 떴을 때만 실행하세요.
#    정상이라면 이 셀은 건너뛰어도 됩니다.
#
# 한국어에 특화된 문장 임베딩 모델을 내 컴퓨터에서 직접 돌린다.
# API 호출이 없으므로 비용이 들지 않고, 인터넷 없이도 동작한다.

# !pip install -q sentence-transformers langchain-huggingface

# from langchain_huggingface import HuggingFaceEmbeddings
#
# embeddings = HuggingFaceEmbeddings(
#     model_name='jhgan/ko-sroberta-multitask',   # 한국어 문장 임베딩 모델
#     encode_kwargs={'normalize_embeddings': True},
# )
#
# v = embeddings.embed_query('연결 테스트')
# print(f'✅ 로컬 임베딩 준비 완료 | 벡터 차원: {len(v)}')
# print('   이후 셀은 수정 없이 그대로 진행하면 됩니다.')

print('이 셀은 임베딩 연결이 실패했을 때만 주석을 해제하여 사용하세요.')

In [ ]:
# ========== 데이터 파일 경로 확인 ==========
# PDF 파일들이 data/ 폴더에 있어야 한다.
#
# 경로는 문자열이 아니라 pathlib.Path 객체로 다룬다.
#   - '/' 연산자로 경로를 이어 붙일 수 있다      : DATA_DIR / 'sample.pdf'
#   - 운영체제(Windows / macOS / Linux)에 관계없이 동작한다
#   - .glob(), .exists(), .name 등 편의 기능이 내장되어 있다
DATA_DIR = Path('data')

# .glob('*.pdf') : 폴더 안의 모든 PDF를 찾는다 (제너레이터를 반환하므로 sorted로 정렬)
pdf_files = sorted(DATA_DIR.glob('*.pdf'))

print(f'PDF 파일 수: {len(pdf_files)}개')
for f in pdf_files:
    print(f'  {f.name}')      # .name : 경로에서 파일명만 꺼낸다 (os.path.basename과 동일)

if not pdf_files:
    print('\n⚠️ data/ 폴더에 PDF 파일이 없습니다.')
    print('  실습 폴더 내 data/ 디렉토리에 Yes24 PDF 파일을 넣어주세요.')
    print(f'  (현재 확인한 경로: {DATA_DIR.resolve()})')

---

## 1. LLM의 한계 — 왜 LLM만으로는 부족한가?

![Image F](https://i.ibb.co/Fbk5qXyS/image-f.png)

### 1-1. LLM의 세 가지 근본적 한계

LLM(Large Language Model)은 방대한 텍스트 데이터로 학습되어 다양한 질문에 답변할 수 있다.
하지만 다음과 같은 **근본적인 한계**가 있다.

| 한계 | 설명 | 예시 |
|------|------|------|
| **환각 (Hallucination)** | 학습 데이터에 없는 내용을 **그럴듯하게 지어냄** | 존재하지 않는 배송 정책을 자신 있게 설명 |
| **최신 정보 부재** | 학습 이후의 정보를 알 수 없음 | 2024년 변경된 정책 미반영 |
| **도메인 지식 부족** | 특정 회사/서비스의 내부 정보는 학습되지 않음 | Yes24 총알배송의 구체적 조건 |

<br>

### 1-2. 환각은 왜 발생하는가?

환각을 "LLM이 거짓말을 한다"고 이해하면 대응 방법을 찾을 수 없다.
**LLM의 동작 원리를 알면 환각이 필연적**이라는 것을 이해할 수 있다.

> **💡 LLM은 '검색'하지 않는다. '다음 단어를 예측'할 뿐이다.**
>
> 2-1 챕터에서 배운 것을 떠올려 보자. LLM은 앞의 문맥을 보고
> **가장 그럴듯한 다음 토큰**을 계속 이어 붙이는 방식으로 문장을 만든다.
>
> ```
> "Yes24의 총알배송 마감 시간은"  ->  다음에 올 법한 말은?
>                                     "오후" (그럴듯함) -> "3시" (그럴듯함) -> ...
> ```
>
> 여기서 모델은 **"내가 이걸 실제로 아는가?"를 확인하지 않는다.**
> 그저 **문장으로서 자연스러운 것**을 고를 뿐이다.
> 그래서 **모르는 것도 유창하게** 말하게 된다.

이 특성 때문에 환각은 아래와 같은 성질을 갖는다.

| 성질 | 설명 | 왜 위험한가 |
|---|---|---|
| **유창하다** | 문법적으로 완벽하고 자신감 있는 문장 | 틀렸다는 신호가 겉으로 드러나지 않음 |
| **그럴듯하다** | 실제 정책과 비슷한 형태로 지어냄 | 전문가가 아니면 구분이 어려움 |
| **일관성이 없다** | 같은 질문을 다시 하면 다른 답이 나오기도 함 | 재현이 안 되어 검증이 어려움 |

<br>

> **⚠️ 2-2 챕터의 그 교훈이 다시 나온다**
>
> 2-2에서 교차 엔트로피를 배우며 이런 말을 했다.
> **"모르면 모른다고 해라. 확신하고 틀리는 게 제일 나쁘다."**
>
> 환각이 정확히 그 상황이다. 고객 서비스에서 이런 답변이 나가면
> 고객 신뢰를 잃고, 법적 문제까지 발생할 수 있다.

### 1-3. 해결 방법은 무엇이 있는가?

환각과 지식 부족을 해결하는 방법은 크게 세 가지다.

| 방법 | 원리 | 비용 | 최신 정보 반영 | 근거 제시 |
|---|---|---|---|:---:|
| **프롬프트 엔지니어링** | 질문을 잘 던진다 | 낮음 | ❌ | ❌ |
| **파인튜닝 (Fine-tuning)** | 모델 자체를 우리 데이터로 추가 학습 | **높음** | ❌ (재학습 필요) | ❌ |
| **RAG** | 답변 전에 관련 자료를 찾아서 읽힌다 | 중간 | ✅ **문서만 갱신** | ✅ **출처 제시 가능** |

<br>

> **💡 파인튜닝 vs RAG — 실무에서 자주 나오는 질문**
>
> | | 파인튜닝 | RAG |
> |---|---|---|
> | 비유 | 사람을 **교육시킨다** | 사람에게 **자료를 쥐여준다** |
> | 잘하는 것 | 말투·형식·스타일 학습 | **사실 정보 제공** |
> | 정보가 바뀌면 | 다시 학습해야 함 | **문서만 교체하면 끝** |
> | 출처 | 알 수 없음 | **어느 문서에서 왔는지 추적 가능** |
>
> 👉 **"우리 회사 정책을 정확히 답해야 한다"** 면 RAG가 정답이다.
> **"우리 회사 말투로 답하게 하고 싶다"** 면 파인튜닝이 어울린다.
> 실무에서는 **둘을 함께 쓰기도** 한다.

아래에서 LLM에게 Yes24 서비스 정책을 직접 물어보고 환각이 발생하는지 확인해 보자.


In [ ]:
from langchain_core.messages import HumanMessage

# ========== LLM에게 직접 질문하기 ==========
# Yes24의 "총알배송"과 "배송지연 보상제도"는 회사 내부 정책이므로
# LLM의 학습 데이터에 정확히 포함되어 있지 않을 가능성이 높다.
questions = [
    'Yes24에서 총알배송이 뭔가요? 서울에서 당일배송 주문 마감 시간은?',
    'Yes24 배송지연 보상제도는 어떻게 되나요? 보상 금액은?',
]

for q in questions:
    response = llm.invoke([HumanMessage(content=q)])
    print(f'질문: {q}')
    print(f'응답: {response.content[1]['text'][:300]}...')
    print()

print('⚠️ 위 답변이 실제 Yes24 정책과 일치하는지 확인할 방법이 없다.')
print('   → 서울 당일배송 마감은 실제로 0~13시 (PDF 참조)')
print('   → 배송지연 보상은 주문 건당 YES포인트 2,000원 (PDF 참조)')


> **🔍 결과를 이렇게 읽어보자**
>
> | 확인할 것 | 왜 중요한가 |
> |---|---|
> | 답변이 **모호한가, 구체적인가** | 구체적인 숫자를 댔다면 그것이 곧 환각 신호다 |
> | **"모르겠다"고 했는가** | 모른다고 말하는 모델이 오히려 안전하다 |
> | 실제 정답과 **얼마나 다른가** | 서울 마감 0~13시 / 보상 2,000P (PDF 기준) |
>
> ⚠️ 모델이 "정확한 정보는 Yes24 공식 페이지를 확인하세요"처럼 **답변을 회피**할 수도 있다.
> 이것도 **좋은 관찰 결과**다. 최신 모델일수록 모르는 것을 인정하도록 학습되어 있기 때문이다.
> 하지만 **회피 역시 우리가 원하는 답이 아니다.** 우리는 정확한 답을 원한다.
>
> 👉 **환각이든 회피든, 결론은 같다. LLM 혼자서는 우리 회사 정책을 답할 수 없다.**


---

## 2. RAG란 무엇인가?

### 2-1. RAG의 정의

**RAG (Retrieval-Augmented Generation)** = 검색 증강 생성

LLM의 환각 문제를 해결하는 가장 효과적인 방법 중 하나이다.
핵심 아이디어는 단순하다: **"답변하기 전에, 먼저 관련 자료를 찾아서 읽고 답변하라."**

```
┌─ 기존 방식 (LLM만) ───────────────────────────────────┐
│  질문 ───→ LLM ───→ 답변 (환각 위험!)                 │
└───────────────────────────────────────────────────────┘

┌─ RAG 방식 ────────────────────────────────────────────┐
│  질문 ───→ [검색] ───→ 관련 문서 ───→ LLM ───→ 답변   │
│                    "이 자료를 참고해서"               │
└───────────────────────────────────────────────────────┘
```

> **💡 비유: 오픈북 시험**
>
> - **LLM만** = 책 없이 시험 보기 → 기억이 부정확하면 **그럴듯하게 지어냄**
> - **RAG** = 오픈북 시험 → 답을 모르면 **책에서 찾아서** 답변 → 정확도 대폭 향상

**이름을 뜯어보면 구조가 보인다.**

| 단어 | 의미 | 담당 |
|---|---|---|
| **R**etrieval (검색) | 질문과 관련된 문서를 찾는다 | Vector Store + Retriever |
| **A**ugmented (증강) | 찾은 문서를 프롬프트에 **덧붙인다** | 프롬프트 템플릿 |
| **G**eneration (생성) | 그 자료를 근거로 답변을 만든다 | LLM |

<br>

> **📌 RAG에서 LLM의 역할이 바뀐다**
>
> RAG를 쓰면 LLM은 **"지식의 원천"이 아니라 "자료를 읽고 정리하는 사람"** 이 된다.
> 그래서 **거대한 모델이 아니어도 된다.** 우리가 `gpt-5-nano`라는 작은 모델로도
> 정확한 답을 얻을 수 있는 이유가 여기에 있다.

<br>

### 2-2. RAG 파이프라인의 전체 구조

RAG는 크게 **준비 단계(Indexing)**와 **실행 단계(Querying)** 두 가지로 나뉜다.

![이미지_A](https://i.ibb.co/jvXKKVt8/image-a.png)


```
┌──────────────── 준비 단계 (Indexing, 1회) ───────────────────┐
│                                                              │
│  [문서 수집]  →  [청킹]  →  [임베딩]  →  [Vector Store 저장] │
│   PDF 로드      잘게 자름    벡터 변환     DB에 저장         │
│                                                              │
└──────────────────────────────────────────────────────────────┘

┌──────────────── 실행 단계 (Querying, 매 질문마다) ─────────────┐
│                                                                │
│  [사용자 질문]  →  [검색]  →  [프롬프트 구성]  →  [LLM 생성]   │
│   "총알배송?"    유사 문서      자료+질문 결합    정확한 답변  │
│                                                                │
└────────────────────────────────────────────────────────────────┘
```

> **⭐ 두 단계를 구분하는 것이 중요한 이유**
>
> | | 준비 단계 | 실행 단계 |
> |---|---|---|
> | 실행 빈도 | **문서가 바뀔 때만 1회** | **질문할 때마다 매번** |
> | 소요 시간 | 길다 (문서가 많으면 수 시간) | 짧다 (수 초) |
> | 비용 | 문서 전체를 임베딩 (일회성) | 질문 1개만 임베딩 (저렴) |
>
> 이 구분을 이해하면 **"왜 미리 Vector Store를 만들어 두는가"** 에 답할 수 있다.
> 질문할 때마다 8개 PDF를 전부 임베딩한다면 너무 느리고 비싸기 때문이다.

### 2-3. RAG의 핵심 구성요소

| 구성요소 | 역할 | 이 실습에서 사용하는 도구 |
|---------|------|:---:|
| **Document Loader** | 문서 파일(PDF, HTML 등)을 텍스트로 변환 | `PyMuPDF4LLMLoader` |
| **Text Splitter** | 긴 문서를 적절한 크기의 조각(chunk)으로 분할 | `RecursiveCharacterTextSplitter` |
| **Embedding Model** | 텍스트를 숫자 벡터로 변환 | `text-embedding-3-small` |
| **Vector Store** | 벡터를 저장하고 유사도 검색 수행 | `ChromaDB` |
| **Retriever** | 질문과 유사한 문서를 Vector Store에서 검색 | `vectorstore.as_retriever()` |
| **LLM** | 검색된 문서를 참고하여 최종 답변 생성 | `gpt-5-nano` (SSAFY GMS) |

<br>

> **💡 다섯 개 부품이 조립되는 순서**
>
> ```
>   Loader  →  Splitter  →  Embedding  →  Vector Store   [준비]
>                                              ↓
>                    질문  →  Retriever  →  LLM          [실행]
> ```
> 이 노트북은 **이 순서를 그대로 따라간다.** 3장부터 하나씩 만들어 갈 것이다.

### 2-4. RAG가 사용되는 실무 사례

| 분야 | 활용 사례 | 왜 RAG가 필요한가 |
|------|---------|------------------|
| **고객 서비스** | 사내 정책 기반 챗봇 (이 실습) | 사내 문서는 LLM 학습 데이터에 없음 |
| **법률** | 판례/법령 검색 기반 법률 자문 | 최신 법률 개정 사항 반영 필요 |
| **의료** | 의학 논문 기반 진단 보조 | 환각으로 인한 오진 방지 |
| **사내 검색** | 사내 위키/문서 기반 Q&A | 기업 내부 정보는 외부 LLM이 알 수 없음 |
| **교육** | 교재 기반 학습 도우미 | 특정 교재의 내용을 정확히 참조해야 함 |

<br>

> **⚠️ RAG도 만능은 아니다 — 한계를 미리 알아두자**
>
> | 한계 | 설명 |
> |---|---|
> | **검색이 틀리면 답도 틀린다** | 엉뚱한 문서를 찾아오면 LLM은 그걸 근거로 자신 있게 틀린다 |
> | **문서에 없는 건 못 답한다** | 자료의 품질이 곧 답변의 상한선이다 |
> | **여러 문서를 종합하는 추론은 약하다** | "A와 B를 비교해줘" 같은 질문은 어렵다 |
> | **비용과 지연이 늘어난다** | 검색 + 긴 프롬프트 = 토큰 증가 |
>
> 👉 **"Garbage In, Garbage Out"** — RAG에서 가장 중요한 것은 화려한 모델이 아니라
> **좋은 문서와 정확한 검색**이다. 그래서 이 노트북의 3~5장이 검색에 집중한다.


### 2-5. RAG의 효과 — 자료를 주면 달라진다

실제로 관련 자료를 프롬프트에 포함시키면 답변이 어떻게 달라지는지 확인해 보자.


In [ ]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_core.prompts import ChatPromptTemplate

# ========== PDF 문서 로드 ==========
# Yes24 총알배송 정책이 담긴 실제 문서를 읽어온다.
# 이 문서가 LLM에게 "오픈북"의 역할을 하게 된다.
#
# PyMuPDF4LLMLoader: PDF를 'Markdown 형식'으로 추출하는 로더.
#   일반 로더는 글자만 뽑아내지만, 이 로더는 제목(#)·목록·표 구조를 살려서 읽어온다.
#   -> 문서의 구조가 보존되므로 LLM이 내용을 더 잘 이해한다. (RAG에 유리)

# 총알배송 관련 PDF 파일 찾기
# f.name(파일명)에 '총알배송'이 포함된 파일만 고른다
bullet_pdf = [f for f in pdf_files if '총알배송' in f.name]

if bullet_pdf:
    # PyMuPDF4LLMLoader는 경로를 문자열로 받으므로 str()로 변환한다
    loader = PyMuPDF4LLMLoader(str(bullet_pdf[0]))
    documents = loader.load()          # 페이지 단위 Document 리스트를 반환

    # 여러 페이지를 하나의 텍스트로 합친다
    doc_content = '\n'.join([doc.page_content for doc in documents])

    print(f'문서 로드 완료: {bullet_pdf[0].name}')
    print(f'페이지 수: {len(documents)}쪽 / 문서 길이: {len(doc_content)}자')
    print(f'\n내용 미리보기:\n{doc_content[:400]}...')
else:
    print('⚠️ 총알배송 PDF를 찾을 수 없습니다.')

In [ ]:
# ========== 자료를 포함하여 질문하기 (RAG의 핵심) ==========
# 시스템 프롬프트에 "참고 자료"를 포함시킨다.
# LLM은 이 자료를 기반으로 답변하므로 환각이 대폭 감소한다.
prompt_template = ChatPromptTemplate.from_messages([
    ('system', '''당신은 온라인 서점 Yes24의 고객 서비스 상담원입니다.
다음 자료를 참고하여 고객의 질문에 정확하게 답변해주세요.
자료에 없는 내용은 "해당 정보는 제공된 자료에 없습니다"라고 답변하세요.

[참고 자료]
{context}'''),
    ('human', '{question}')
])

question = 'Yes24에서 총알배송이 뭔가요? 서울에서 당일배송 주문 마감 시간은?'
messages = prompt_template.format_messages(context=doc_content, question=question)
response = llm.invoke(messages)

print(f'질문: {question}')
print(f'\n자료 기반 응답:')
print(response.content[1]['text'][:500])
print('\n✅ 실제 PDF 문서에 기반한 정확한 답변!')


### 2-6. 그런데... 자료는 어떻게 "자동으로" 찾는가?

방금 실습에서는 **정답 문서를 우리가 직접 골라서** 프롬프트에 넣었다.
하지만 실제 서비스에서는 **수백~수천 개의 문서** 중에서 관련 문서를 **자동으로 찾아야** 한다.

```
사용자: "배송이 늦으면 보상받을 수 있나요?"

    → 총알배송 PDF? 배송지연 보상 PDF? 도서 품절 보상 PDF? 신규 회원 PDF?
      8개의 PDF 중 어떤 것이 관련 있는지 자동으로 판단해야 한다!
```

이 **"자동 검색"** 문제를 해결하는 것이 RAG 파이프라인의 핵심이다.
다음 챕터부터 검색 방법을 단계적으로 배워 보자.


---

## 3. 검색의 진화 — 키워드에서 의미로

RAG의 성능은 **검색이 결정한다.** 이번 장에서는 검색 방식을 단계적으로 발전시켜 본다.

### 3-1. 키워드 검색 (Lexical Search)

가장 단순한 검색 방법: **문자열이 정확히 포함되어 있는지** 확인한다.

```python
if '총알배송' in document:
    return document  # 문자열이 포함되면 반환
```

| 장점 | 단점 |
|------|------|
| 단순하고 빠르다 | **정확히 일치하는 키워드만** 찾을 수 있다 |
| 구현이 쉽다 | 동의어, 유사 표현을 전혀 인식하지 못한다 |

<br>

> **💡 핵심 문제**
>
> 고객이 "빠른 배송"이라고 질문하면 "총알배송" 문서를 **찾지 못한다.**
> 고객이 "포인트 유효기간"이라고 질문하면 "영원한 YES포인트" 문서를 **찾지 못한다.**
> 사람은 둘이 같은 의미인 걸 바로 알지만, 키워드 검색은 **글자가 다르면 다른 것**으로 취급한다.

> **📌 실무의 키워드 검색은 조금 더 정교하다 — BM25**
>
> 아래 실습 코드는 `in` 연산자를 쓰는 가장 단순한 형태다.
> 실무에서는 보통 **BM25**라는 알고리즘을 쓴다.
>
> - 흔한 단어("은/는/이/가")는 **가중치를 낮추고**
> - 드문 단어("총알배송")는 **가중치를 높인다**
> - 문서 길이도 보정한다
>
> 하지만 **BM25도 결국 '글자'를 본다.** "빠른 배송"과 "총알배송"이 다른 단어라는
> 근본적인 한계는 그대로다. 그래서 의미 기반 검색이 필요하다.

<br>

> **⚠️ 그렇다고 키워드 검색이 쓸모없는 것은 아니다**
>
> 오히려 키워드 검색이 **더 강한 경우**가 있다.
>
> | 상황 | 왜 키워드가 유리한가 |
> |---|---|
> | 상품 코드, 주문번호 검색 | `A-2024-0417` 은 의미가 없다. **정확히 일치**해야 한다 |
> | 고유명사, 사람 이름 | 임베딩이 비슷한 이름을 헷갈릴 수 있다 |
> | 법조문 번호, 약어 | 정확성이 절대적으로 중요하다 |
>
> 그래서 실무에서는 **두 방식을 함께 쓰는 하이브리드 검색(Hybrid Search)** 이 표준이다.
> 키워드 검색 결과와 의미 검색 결과를 합쳐서 순위를 다시 매기는 방식이다.
>
> 👉 이 실습에서는 **개념을 명확히 하기 위해** 두 방식을 따로 비교한다.


> **📌 이 실습이 쓰는 PDF 로더 — `PyMuPDF4LLMLoader`**
>
> PDF에서 텍스트를 뽑는 도구는 여러 가지가 있다. 각각 특성이 다르다.
>
> | 로더 | 특징 | 적합한 문서 |
> |---|---|---|
> | `PyPDFLoader` | 가장 기본. 빠르지만 구조 정보가 사라짐 | 단순 텍스트 PDF |
> | `PyMuPDFLoader` | 빠르고 안정적. 평문 텍스트를 반환 | 일반적인 문서 |
> | **`PyMuPDF4LLMLoader`** | **Markdown 형식**으로 추출 (제목·목록·표 보존) | **RAG용** ← 이 실습 |
> | `UnstructuredPDFLoader` | 레이아웃 분석이 정교하지만 무겁고 느림 | 복잡한 레이아웃 |
>
> **왜 Markdown 형식이 RAG에 유리한가?**
>
> ```
>   [평문 추출]                      [Markdown 추출]
>   총알배송 안내                     # 총알배송 안내
>   서울 0~13시                       - 서울: 0~13시
>   경기 0~11시                       - 경기: 0~11시
>
>   -> 그냥 줄바꿈된 글자             -> 제목과 목록이라는 '구조'가 남는다
> ```
>
> 구조가 남으면 두 가지 이득이 있다.
> 1. **LLM이 내용을 더 잘 이해한다** — 무엇이 제목이고 무엇이 항목인지 구분된다
> 2. **청킹할 때 유리하다** — 제목 단위로 자르는 전략을 쓸 수 있다 (5장 참고)
>
> ⚠️ 다만 **스캔한 이미지 PDF**는 어떤 로더로도 텍스트를 못 뽑는다. 이 경우 OCR이 필요하다.


In [ ]:
# ========== 모든 PDF 문서 로드 ==========
# 앞에서는 총알배송 PDF 하나만 읽었지만,
# 이제 검색 실습을 위해 data/ 폴더의 PDF 전체를 읽어온다.
all_documents = []
for pdf_path in pdf_files:
    loader = PyMuPDF4LLMLoader(str(pdf_path))
    docs = loader.load()               # PDF 1개 -> 페이지 수만큼의 Document
    all_documents.extend(docs)         # 리스트에 이어 붙인다 (append가 아니라 extend)

print(f'로드된 문서 수: {len(all_documents)}개 (PDF {len(pdf_files)}개)')

# ========== 키워드 검색 함수 ==========
def keyword_search(documents, keyword):
    """문서 리스트에서 키워드를 포함한 문서를 검색한다.

    in 연산자로 '문자열이 그대로 들어 있는지'만 확인한다.
    -> 글자가 조금이라도 다르면 찾지 못한다는 한계가 여기서 생긴다.
    """
    return [doc for doc in documents if keyword in doc.page_content]

# ========== 키워드 검색의 한계 테스트 ==========
# 같은 의미이지만 다른 표현으로 검색해 본다.
test_keywords = [
    ('총알배송', '정확한 키워드'),
    ('빠른 배송', '동의어'),
    ('배송 빨리', '유사 표현'),
    ('포인트 유효기간', '관련 표현 → "영원한 YES포인트"를 찾을 수 있을까?'),
]

print('\n키워드 검색 결과:')
for kw, desc in test_keywords:
    results = keyword_search(all_documents, kw)
    status = '✅' if results else '❌'
    print(f'  {status} "{kw}" ({desc}) → {len(results)}건')

print('\n→ 정확한 키워드만 찾고, 동의어/유사 표현은 전혀 인식하지 못한다.')

### 3-2. 의미 기반 검색 (Semantic Search)

키워드 검색의 한계를 극복하는 방법: **텍스트의 "의미"를 이해하는 검색.**

![image C](https://i.ibb.co/fdkcrxXD/image-c.png)

핵심 원리:
1. 모든 문서를 **숫자 벡터(임베딩)**로 변환하여 저장
2. 사용자의 질문도 **같은 방식으로 벡터로 변환**
3. 질문 벡터와 **가장 가까운(유사한) 문서 벡터**를 찾아 반환

```
"빠른 배송"   → [0.2, -0.5, 0.8, ...]  ← 질문 벡터
"총알배송"    → [0.3, -0.4, 0.7, ...]  ← 문서 벡터 (의미가 비슷 → 벡터도 비슷!)
"포인트 적립" → [0.9, 0.1, -0.3, ...]  ← 관련 없는 문서 (벡터가 멀다)
```

> **💡 2-1 챕터 연결**
>
> 2-1에서 "영화"와 유사한 단어를 벡터로 찾았던 것과 동일한 원리이다.
> 차이점: 이제 **단어가 아닌 문서 전체**를 벡터로 변환한다.

### 3-3. 두 방식 비교

| 항목 | 키워드 검색 | 의미 기반 검색 |
|---|---|---|
| 비교 대상 | **글자**가 같은가 | **의미**가 가까운가 |
| 동의어 | ❌ 인식 못 함 | ✅ 인식함 |
| 오타 | ❌ 취약 | ✅ 어느 정도 견딤 |
| 고유명사·코드 | ✅ 정확 | ⚠️ 헷갈릴 수 있음 |
| 속도 | 매우 빠름 | 빠름 (벡터 연산) |
| 준비 작업 | 없음 | **임베딩 + Vector Store 구축 필요** |
| 비용 | 없음 | 임베딩 API 비용 발생 |

<br>

> **⭐ 왜 의미 검색은 '준비 작업'이 필요한가?**
>
> 질문할 때마다 문서 전체를 임베딩하면 너무 느리고 비싸다.
> 그래서 **미리 문서를 전부 벡터로 만들어 DB에 저장**해 둔다.
> 이것이 2-2절에서 본 **준비 단계(Indexing)** 다.
>
> 다음 장에서 그 DB, 즉 **Vector Store**를 직접 만들어 본다.


---

## 4. 임베딩과 Vector Store

### 4-1. 임베딩(Embedding) 복습

2-1 챕터에서 배운 핵심:
- 텍스트를 **고차원 벡터**(예: 1536차원)로 변환하는 기술
- 의미가 비슷한 텍스트는 **벡터 공간에서 가까운 위치**에 놓인다

| 구분 | 2-1 챕터 | 이번 챕터 |
|------|:---:|:---:|
| 대상 | 단어 단위 | 문장/문서 단위 |
| 모델 | `nn.Embedding` (768차원) | `text-embedding-3-small` (1536차원, GMS 경유) |
| 용도 | 유사 단어 찾기 | **유사 문서 찾기 (RAG 검색)** |

![image D](https://i.ibb.co/1YnhvcxX/image-d.png)

> **💡 2-1에서 배운 '두 가지 임베딩'을 기억하는가?**
>
> | | 임베딩 층 (입력) | 문맥 임베딩 (출력) |
> |---|---|---|
> | 정체 | 단어 -> 벡터 변환표 | 모델을 다 통과한 결과 |
> | 문맥 반영 | ❌ | ✅ |
>
> 이번에 쓰는 `text-embedding-3-small`은 **후자에 가깝다.**
> 문장 전체를 읽고 **문맥이 반영된 벡터 하나**를 뱉는다.
> 그래서 "빠른 배송"과 "총알배송"이 가까운 벡터가 될 수 있는 것이다.

### 4-2. 코사인 유사도 — 왜 '거리'가 아니라 '각도'인가

두 벡터가 얼마나 비슷한지 재는 방법은 여러 가지다.
그중 텍스트 임베딩에서는 거의 항상 **코사인 유사도**를 쓴다.

```
        ↗ A          두 벡터가 이루는 '각도'만 본다
       /             각도가 작을수록  ->  유사도 1에 가까움
      /  θ           각도가 90도면    ->  유사도 0 (무관)
     /____→ B        각도가 180도면   ->  유사도 -1 (반대)
    O
```

| 측정 방법 | 무엇을 보는가 | 텍스트에 적합한가 |
|---|---|---|
| 유클리드 거리 | 두 점 사이의 **직선 거리** | ⚠️ 문서 길이에 영향받음 |
| **코사인 유사도** | 두 벡터의 **방향(각도)** | ✅ **길이와 무관** |

<br>

> **⭐ 왜 길이와 무관한 것이 중요한가?**
>
> 같은 내용이라도 **긴 문서는 벡터의 크기(길이)가 커지는 경향**이 있다.
> 유클리드 거리를 쓰면 "긴 문서끼리 비슷하다"는 엉뚱한 결론이 나온다.
> 코사인 유사도는 **각도만 보므로 이 문제가 없다.**
>
> 2-1 챕터에서 유사 단어를 찾을 때도 코사인 유사도를 썼던 이유가 같다.

**유사도 값 읽는 법**

| 값 | 의미 |
|---|---|
| 0.9 이상 | 거의 같은 내용 |
| 0.7 ~ 0.9 | 관련이 깊음 |
| 0.5 ~ 0.7 | 어느 정도 관련 |
| 0.5 미만 | 관련이 약함 |

<br>

> ⚠️ 이 기준은 **모델마다 다르다.** 절대값보다 **같은 질문 안에서의 순위**를 보는 것이 안전하다.

실제 임베딩 벡터를 생성하고, Yes24 문서 관련 문장들의 유사도를 계산해 보자.


In [ ]:
import numpy as np

# ========== 임베딩 벡터 생성 + 유사도 계산 ==========
texts = [
    '빠른 배송 서비스가 궁금합니다',           # 질문
    '총알배송 표시상품을 주문하시면 당일에 받으실 수 있습니다',  # 관련 문서 (총알배송)
    'YES포인트는 유효기간이 없습니다',          # 관련 없는 문서 (포인트)
    '배송이 지연되면 보상을 받을 수 있나요',     # 다른 주제 (배송지연)
]

vectors = embeddings.embed_documents(texts)
print(f'벡터 차원: {len(vectors[0])}차원')

# 코사인 유사도 계산
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query_vec = vectors[0]  # "빠른 배송 서비스가 궁금합니다"
print(f'\n기준: "{texts[0]}"')
for i in range(1, len(texts)):
    sim = cosine_similarity(query_vec, vectors[i])
    print(f'  vs "{texts[i][:25]}..."  유사도: {sim:.4f}')

print('\n→ "총알배송"이 "빠른 배송"과 가장 유사하게 나온다!')
print('   키워드 검색에서는 불가능했던 것이 의미 기반 검색에서는 가능하다.')


> **🔍 결과에서 확인할 것**
>
> 1. **"총알배송"이 1위인가?** — 글자는 하나도 안 겹치는데 의미로 찾아냈다면 성공이다
> 2. **"배송이 지연되면"은 몇 위인가?** — 같은 '배송' 주제라 중간쯤 나올 것이다
> 3. **"YES포인트"가 꼴찌인가?** — 완전히 다른 주제이므로 가장 낮아야 정상이다
>
> 💡 값들이 전부 0.3~0.8 사이에 몰려 있어도 정상이다.
> **중요한 것은 절대값이 아니라 순서**다.

> **🧪 직접 실험해 보기**
>
> `texts` 리스트에 문장을 추가해서 유사도가 어떻게 변하는지 관찰해 보자.
>
> ```python
> texts = [
>     '빠른 배송 서비스가 궁금합니다',
>     '총알배송 표시상품을 주문하시면 당일에 받으실 수 있습니다',
>     '오늘 주문하면 언제 도착하나요',        # 추가: 같은 의미, 완전히 다른 표현
>     '내일 날씨가 어떤가요',                 # 추가: 완전히 무관한 문장
> ]
> ```
> **글자가 하나도 안 겹쳐도** 의미가 비슷하면 유사도가 높게 나오는지 확인해 보자.


### 4-3. Vector Store란?

모든 문서의 임베딩 벡터를 **저장하고 빠르게 검색**할 수 있는 특수 데이터베이스이다.

```
[Vector Store 동작 흐름]

① 저장 (인덱싱)  —  1회만 실행
   문서들 → 임베딩 모델 → 벡터들 → Vector Store에 저장

② 검색  —  매 질문마다 실행
   질문 → 임베딩 모델 → 질문 벡터 → Vector Store에서 유사한 벡터 검색 → 관련 문서 반환
```

> **💡 일반 DB와 무엇이 다른가?**
>
> | | 일반 DB (RDB) | Vector Store |
> |---|---|---|
> | 저장 대상 | 행과 열로 된 정형 데이터 | **고차원 벡터** |
> | 검색 방식 | `WHERE name = '총알배송'` (**정확히 일치**) | **가장 가까운 벡터 k개** |
> | 질문 형태 | "이 값과 같은 것" | "이것과 비슷한 것" |
>
> 👉 SQL로는 "비슷한 것을 찾아줘"라는 질문 자체가 불가능하다. 그래서 새로운 DB가 필요했다.

> **⭐ 벡터 1만 개를 전부 비교하면 느리지 않을까?**
>
> 맞다. 그래서 Vector Store는 **ANN(Approximate Nearest Neighbor, 근사 최근접 탐색)** 이라는
> 기법을 쓴다.
>
> - **정확한 방법**: 1만 개와 전부 비교 -> 정확하지만 느림
> - **ANN**: 미리 만들어 둔 색인으로 **후보를 좁혀서** 비교 -> 아주 빠르고, 정확도는 거의 동일
>
> "약간의 정확도를 내주고 속도를 크게 얻는" 전략이다.
> 문서가 수백만 개가 되어도 검색이 밀리초 단위로 끝나는 이유가 여기에 있다.

대표적인 Vector Store:

| Vector Store | 특징 | 적합한 상황 |
|:---:|------|------|
| **ChromaDB** | 로컬 환경에서 간편하게 사용 | 실습, 프로토타이핑 (이 실습에서 사용) |
| **FAISS** | Meta에서 개발한 고속 검색 | 대규모 데이터, 속도 중시 |
| **Pinecone** | 클라우드 관리형 서비스 | 운영 환경, 확장성 중시 |
| **pgvector** | PostgreSQL 확장 | 기존 RDB와 함께 쓰고 싶을 때 |

<br>

> 📌 실무에서는 디스크에 저장한다
>
> 이 실습은 메모리에만 저장하므로 커널을 재시작하면 사라진다. 실무에서는 `persist_directory`를 지정해 파일로 저장하고, 문서가 바뀔 때만 다시 만든다.
>
> ⚠️ 단, `from_documents`를 다시 호출하면 데이터가 중복 누적되므로, 기존 DB가 있으면 불러오기만 하도록 분기해야 한다.

<br>

> **📌 `k` 값은 무엇인가**
>
> 아래 코드의 `search_kwargs={'k': 3}` 는 **"가장 비슷한 문서 3개를 가져와라"** 는 뜻이다.
>
> | k가 작으면 | k가 크면 |
> |---|---|
> | 정확한 자료만 전달 | 관련 자료를 놓칠 확률 감소 |
> | 정답 문서를 놓칠 위험 | **노이즈가 섞임** + 토큰 비용 증가 |
>
> 보통 **3~5**에서 시작해 조정한다. 이 실습 마지막에 직접 바꿔보는 실험이 있다.


In [ ]:
from langchain_community.vectorstores import Chroma
import tiktoken

# ========== 토큰 제한 처리 ==========
# 임베딩 모델(text-embedding-3-small)은 최대 8,191 토큰까지 처리 가능하다.
# PDF 전체 페이지가 이 한도를 초과할 수 있으므로 미리 잘라 준다.
MAX_TOKENS = 8000
enc = tiktoken.encoding_for_model('text-embedding-3-small')

truncated = 0
for doc in all_documents:
    tokens = enc.encode(doc.page_content)
    if len(tokens) > MAX_TOKENS:
        doc.page_content = enc.decode(tokens[:MAX_TOKENS])
        truncated += 1
if truncated:
    print(f'⚠️ {truncated}개 문서가 토큰 제한 초과로 잘림')

# ========== Vector Store 생성 (페이지 단위) ==========
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name='yes24_page',
    # persist_directory=str(Path('chroma_db')),
)
print(f'Vector Store 생성 완료 (저장: {vectorstore._collection.count()}개)')

# Retriever 생성: 상위 3개 유사 문서를 반환
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})


In [ ]:
# ========== 의미 기반 검색 테스트 ==========
# 키워드 검색에서 실패했던 질문들로 테스트한다.
test_queries = [
    ('빠른 배송', '→ "총알배송" 문서를 찾을 수 있는가?'),
    ('포인트 유효기간', '→ "영원한 YES포인트" 문서를 찾을 수 있는가?'),
    ('배송이 늦으면 보상', '→ "배송지연 보상제도" 문서를 찾을 수 있는가?'),
]

for query, desc in test_queries:
    results = retriever.invoke(query)
    # metadata['source']에는 원본 파일의 전체 경로가 들어 있다.
    # Path(...).name 으로 파일명만 꺼낸다.
    source = Path(results[0].metadata.get('source', '')).name if results else '없음'
    print(f'  "{query}" {desc}')
    print(f'    → 검색된 문서: {source}')
    print(f'    → 내용 (앞 200자): {results[0].page_content[:200]}...')
    print()

print('✅ 키워드가 정확히 일치하지 않아도 의미적으로 관련된 문서를 찾아낸다!')


---

## 5. 청킹(Chunking) 전략

### 5-1. 왜 청킹이 필요한가?

![image E](https://i.ibb.co/jkrBdwRd/image-e.png)

지금까지 문서를 **페이지 단위**로 Vector Store에 저장했다.
하지만 한 페이지에 **여러 주제**가 섞여 있을 수 있다.

"배송"을 질문했는데 페이지 전체가 반환되면, 반품/회원 정보까지 LLM에게 전달된다.
이런 **노이즈**는 답변 품질을 떨어뜨린다.

**청킹**: 긴 문서를 적절한 크기의 **조각(chunk)**으로 나누는 기법

> **💡 청킹이 해결하는 세 가지 문제**
>
> | 문제 | 페이지 단위일 때 | 청킹 후 |
> |---|---|---|
> | **검색 정밀도** | 한 페이지에 여러 주제 -> 엉뚱한 페이지가 걸림 | 주제별로 나뉘어 정확히 걸림 |
> | **노이즈** | 관계없는 내용까지 LLM에 전달 | 필요한 부분만 전달 |
> | **토큰 비용** | 페이지 전체가 프롬프트에 들어감 | 조각만 들어감 -> **비용 절감** |
>
> ⚠️ 한 가지 더. 임베딩 모델은 **긴 텍스트일수록 의미가 뭉개진다.**
> 여러 주제가 섞인 페이지를 벡터 하나로 만들면, 그 벡터는 **어느 주제와도 어중간하게** 가까워진다.

<br>

### 5-2. 청킹의 두 가지 파라미터

| 파라미터 | 설명 | 비유 |
|---------|------|------|
| `chunk_size` | 조각의 최대 크기 (글자 수) | 카드 한 장에 적을 수 있는 글자 수 |
| `chunk_overlap` | 조각 간 겹치는 부분 | 카드 사이에 겹쳐 놓는 영역 (문맥 유지) |

<br>

> **⭐ `chunk_overlap`이 왜 필요한가?**
>
> 겹침이 없으면 **문장이 조각 경계에서 잘려나간다.**
>
> ```
>   [겹침 없음]
>   조각1: "...서울 지역 당일배송 마감 시간은"
>   조각2: "0~13시입니다. 경기 지역은..."
>            ↑ 두 조각 어느 쪽도 완전한 답을 갖고 있지 않다!
>
>   [겹침 있음 (overlap=50)]
>   조각1: "...서울 지역 당일배송 마감 시간은 0~13시입니다."
>   조각2: "마감 시간은 0~13시입니다. 경기 지역은..."
>            ↑ 양쪽 모두 완전한 정보를 갖는다
> ```
>
> 보통 `chunk_size`의 **10~20%** 정도를 겹치게 한다.

### 5-3. chunk_size는 어떻게 정하는가

| chunk_size | 장점 | 단점 | 적합한 상황 |
|:---:|------|------|------|
| 작음 (100~200) | 세밀한 검색 | 문맥 부족 | FAQ, 짧은 Q&A |
| 중간 (300~500) | 균형 | - | 정책 문서, 일반 안내 |
| 큼 (1000+) | 풍부한 문맥 | 노이즈 포함 | 논문, 기술 문서 |

<br>

> **💡 정답은 없다. 문서의 성격이 결정한다.**
>
> 판단 기준은 하나다. **"하나의 완결된 정보 단위가 한 조각에 들어가는가?"**
>
> - Yes24 정책 문서 -> 한 항목이 2~3문장 -> **300자가 적절**
> - 논문 -> 한 단락이 길고 앞뒤 문맥이 중요 -> 1000자 이상
> - FAQ -> 질문-답변 한 쌍이 짧음 -> 100~200자
>
> 실무에서는 **여러 값으로 만들어보고 검색 품질을 비교**해서 정한다.

### 5-4. `RecursiveCharacterTextSplitter`가 똑똑한 이유

단순히 300자마다 자르면 **문장 중간이 잘린다.**
`RecursiveCharacterTextSplitter`는 **의미 단위를 지키려고 노력**한다.

```python
separators=['\n\n', '\n', '.', ' ', '']
#            문단      줄     문장   단어  글자
```

이 순서대로 **위에서부터 시도**한다.

```
① 문단(\n\n)으로 잘라본다  ->  조각이 300자 이하면 성공, 끝
② 너무 크면 줄바꿈(\n)으로 더 잘라본다
③ 그래도 크면 마침표(.)로
④ 그래도 크면 공백( )으로
⑤ 최후에는 글자 단위로
```

> 👉 **가능한 한 큰 의미 단위를 유지하면서** 크기를 맞추는 것이 핵심이다.
> 그래서 이름에 **Recursive(재귀적)** 가 붙었다.

> **📌 더 발전된 청킹 전략들 (참고)**
>
> | 전략 | 아이디어 |
> |---|---|
> | **Semantic Chunking** | 글자 수가 아니라 **의미가 바뀌는 지점**에서 자른다 |
> | **Markdown/Header 기반** | 문서의 **제목 구조**를 따라 자른다 |
> | **Parent-Child** | 검색은 작은 조각으로, LLM에게는 **큰 원본**을 전달한다 |
>
> 💡 이 실습의 PDF 로더(`PyMuPDF4LLMLoader`)는 문서를 **Markdown 형식**으로 읽어온다.
> 제목(`#`) 구조가 살아 있으므로, 나중에 Header 기반 청킹으로 확장하기 쉽다.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ========== 청킹 적용 + 새 Vector Store ==========
# Yes24 정책 문서는 조건부 설명이 많으므로 chunk_size=300이 적절하다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '.', ' ', '']  # 이 순서대로 분할 시도
)

chunked_documents = text_splitter.split_documents(all_documents)
print(f'원본: {len(all_documents)}개 → 청킹 후: {len(chunked_documents)}개 조각')

# 청킹된 Vector Store 생성
vectorstore_chunked = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    collection_name='yes24_chunked'
)
retriever_chunked = vectorstore_chunked.as_retriever(search_kwargs={'k': 3})
print(f'청킹 Vector Store 생성 완료 ({vectorstore_chunked._collection.count()}개 벡터)')


In [ ]:
# ========== 페이지 vs 청킹 검색 비교 ==========
query = '배송이 늦으면 보상받을 수 있나요?'

results_page = retriever.invoke(query)
results_chunk = retriever_chunked.invoke(query)

print(f'질문: {query}\n')
print(f'--- 페이지 단위 검색 (첫 결과: {len(results_page[0].page_content)}자) ---')
print(f'{results_page[0].page_content[:300]}...')

print(f'\n--- 청킹 단위 검색 (첫 결과: {len(results_chunk[0].page_content)}자) ---')
print(f'{results_chunk[0].page_content[:300]}...')

print('\n→ 청킹된 결과가 더 집중적이고 관련성이 높다!')


> **🔍 비교 결과 읽는 법**
>
> | 확인할 것 | 기대되는 결과 |
> |---|---|
> | **글자 수** | 페이지 단위는 수천 자, 청킹은 300자 내외 |
> | **내용의 집중도** | 청킹 쪽이 질문과 직접 관련된 내용으로 시작하는가 |
> | **노이즈** | 페이지 단위에 관계없는 주제가 섞여 있는가 |
>
> ⚠️ 두 결과가 비슷하게 나올 수도 있다. **원본 PDF의 한 페이지가 이미 짧다면** 그렇다.
> 그 경우 청킹의 효과는 문서가 길어질수록 커진다고 이해하면 된다.

> **🧪 직접 실험해 보기**
>
> `chunk_size`를 바꿔가며 검색 결과가 어떻게 달라지는지 확인해 보자.
>
> | 값 | 예상 결과 |
> |---|---|
> | `chunk_size=100` | 조각 수 급증. 검색은 정밀하지만 **문맥이 잘려** 답변이 부실할 수 있다 |
> | `chunk_size=300` | 현재 설정 (균형) |
> | `chunk_size=1000` | 조각 수 감소. 문맥은 풍부하지만 **노이즈**가 섞인다 |
>
> 바꾼 뒤에는 **Vector Store를 다시 만들어야 한다.** (`collection_name`도 함께 바꿀 것)


---

## 6. LangChain과 LangGraph, 그리고 RAG 파이프라인

### 6-1. 지금까지의 여정

```
① LLM만 → 환각 발생 (챕터 1)
② RAG = "답변 전에 자료를 먼저 찾아 읽어라" (챕터 2)
③ 키워드 검색 → 동의어 인식 불가 (챕터 3)
④ 임베딩 + Vector Store → 의미 기반 검색 (챕터 4)
⑤ 청킹으로 검색 정밀도 향상 (챕터 5)
⑥ 이 모든 것을 하나의 파이프라인으로! (이번 챕터)
```

부품은 다 만들었다. 이제 **조립**할 차례다.

<br>

### 6-2. 잠깐 — 우리는 이미 LangChain을 쓰고 있었다

본격적으로 들어가기 전에 짚고 갈 것이 있다.
지금까지 쓴 코드를 다시 보자.

```python
from langchain_openai      import ChatOpenAI, OpenAIEmbeddings
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_community.vectorstores    import Chroma
from langchain_text_splitters            import RecursiveCharacterTextSplitter
from langchain_core.prompts              import ChatPromptTemplate
```

**전부 `langchain`으로 시작한다.** 우리는 처음부터 LangChain 생태계 위에서 실습하고 있었다.

#### LangChain이란?

**LLM 애플리케이션을 만들기 위한 부품 상자**다.
LLM으로 무언가를 만들려면 매번 비슷한 작업이 필요하다.

| 필요한 일 | LangChain이 제공하는 부품 | 이 실습에서 |
|---|---|:---:|
| 여러 LLM을 같은 방식으로 호출 | `ChatOpenAI`, `ChatAnthropic` ... | ✅ |
| PDF·HTML·CSV 등 문서 읽기 | Document Loaders | ✅ |
| 긴 문서 자르기 | Text Splitters | ✅ |
| 텍스트를 벡터로 | Embeddings | ✅ |
| 벡터 DB 연결 | Vector Stores | ✅ |
| 프롬프트 템플릿 관리 | `ChatPromptTemplate` | ✅ |

<br>

> **💡 LangChain이 없다면?**
>
> PDF를 읽는 코드, 자르는 코드, 임베딩 API를 호출하는 코드, ChromaDB에 넣는 코드를
> **전부 직접 짜야 한다.** 그리고 OpenAI에서 다른 모델로 바꾸는 순간 다 고쳐야 한다.
>
> LangChain은 이것들을 **표준 인터페이스로 감싸준다.**
> 그래서 우리가 Upstage에서 GMS로 바꿀 때 `base_url` 한 줄만 고치면 됐던 것이다.

> **⚠️ 이름이 헷갈리기 쉽다 — 이름의 유래**
>
> **Chain(사슬)** = 여러 단계를 **줄줄이 엮는다**는 뜻이다.
> `문서 로드 -> 자르기 -> 임베딩 -> 검색 -> 프롬프트 -> LLM` 처럼
> 작업을 사슬처럼 연결하는 것이 원래 컨셉이었다.

<br>

### 6-3. 그럼 LangGraph는 무엇인가?

**LangChain 팀이 만든, 더 복잡한 흐름을 다루기 위한 도구**다.

이름에서 차이가 드러난다. **Chain(사슬)** vs **Graph(그래프)**.

```
   [LangChain의 Chain]           [LangGraph의 Graph]

   A → B → C → D                     ┌─→ B ─┐
                                 A ──┤      ├─→ D
   한 줄로 쭉 흘러간다                 └─→ C ─┘  │
   되돌아갈 수 없다                    ↑         │
                                      └────────┘
                                   갈라지고, 합쳐지고, 되돌아온다
```

| 구분 | LangChain (Chain) | LangGraph (Graph) |
|---|---|---|
| 흐름 | **한 방향으로 쭉** | 분기·반복·되돌아가기 가능 |
| 조건 분기 | 어렵다 | ✅ **쉽다** |
| 반복(재시도) | 어렵다 | ✅ **쉽다** |
| 상태 관리 | 단계마다 값을 넘김 | ✅ **State에 모아서 공유** |
| 적합한 작업 | 정해진 순서의 단순 작업 | **에이전트, 복잡한 워크플로우** |

<br>

> **💡 왜 그래프가 필요한가 — 실무의 RAG는 직선이 아니다**
>
> 단순한 RAG는 "검색 -> 생성" 2단계라 사슬로도 충분하다.
> 하지만 실무에서는 이런 흐름이 필요하다.
>
> | 상황 | 필요한 동작 |
> |---|---|
> | 검색 결과가 없다 | **다른 DB에서 재검색** (분기) |
> | 답변 품질이 낮다 | **키워드를 바꿔 다시 검색** (반복) |
> | 자료가 여러 곳에 있다 | **동시에 검색 후 합치기** (병렬) |
> | 민감한 답변이다 | **사람의 승인을 받고 진행** (Human-in-the-loop) |
>
> 전부 **직선으로는 표현할 수 없는 흐름**이다. 그래서 그래프가 필요하다.

<br>

### 6-4. 왜 우리는 LangChain을 건너뛰고 LangGraph부터 하는가?

> **📌 이 질문은 아주 자연스럽다. 이유를 명확히 하고 넘어가자.**

**① 이미 LangChain의 핵심 부품은 다 써봤다**

6-2에서 확인했듯 Loader, Splitter, Embeddings, Vector Store, Prompt를 전부 사용했다.
남은 것은 그것들을 **엮는 방법**뿐인데, 그 방법이 LangGraph다.

**② 현재 생태계의 흐름이 LangGraph로 옮겨갔다**

LangChain의 옛 `Chain` 문법(`LLMChain`, `RetrievalQA` 등)은 유지보수 모드에 가깝다.
LangChain 공식 문서도 **복잡한 흐름은 LangGraph로 만들 것을 권장**한다.
지금 배워두면 그대로 실무에서 쓸 수 있다.

**③ RAG는 결국 에이전트로 확장된다**

다음 챕터에서 배울 **AI Agent**는 "생각 -> 도구 사용 -> 결과 관찰 -> 다시 생각"을 반복한다.
이것은 **반복이 있는 그래프**다. Chain으로는 표현할 수 없다.
지금 LangGraph를 익혀두면 그대로 이어진다.

> **📚 LangChain 자체는 자기주도 학습으로**
>
> LangChain의 문법(LCEL, `|` 파이프 연산자, `Runnable` 등)은
> **별도 자기주도 학습 자료**에서 다룬다.
>
> | 여기서 배우는 것 | 자기주도 학습에서 배우는 것 |
> |---|---|
> | LangChain **부품 사용법** (실습으로 체득) | LangChain **문법과 체이닝** (LCEL) |
> | **LangGraph**로 흐름 설계 | `Runnable` 인터페이스, 스트리밍 |
>
> 👉 이 노트북을 먼저 해도, 자기주도 학습을 먼저 해도 무방하다.
> **부품을 알고 조립법을 배우는 순서**가 이 노트북이다.

<br>

### 6-5. LangGraph란?

LangChain 팀에서 개발한 **상태 기반 워크플로우 프레임워크**이다.
AI 애플리케이션의 복잡한 작업 흐름을 **그래프(Graph)** 구조로 설계할 수 있게 해준다.

> **💡 핵심 개념 한 줄 요약**
>
> **"공유되는 칠판(State)을 두고, 여러 작업자(Node)가 정해진 순서(Edge)대로 읽고 쓴다."**


### 6-6. LangGraph의 핵심 구성요소

LangGraph는 세 가지 핵심 요소로 구성된다.

![image B](https://i.ibb.co/84DCh1dK/image-b.png)

#### ① State (상태)

그래프 전체에서 **공유되는 데이터 저장소**이다. 모든 노드가 이 State를 읽고 업데이트한다.

```python
from typing import TypedDict

class MyState(TypedDict):
    question: str    # 사용자 질문
    answer: str      # 생성된 답변
```

- `TypedDict`를 사용하여 어떤 데이터가 오가는지 **타입을 명시**한다
- 비유: **칠판** — 모든 작업자가 같은 칠판을 보며 읽고 쓴다

#### ② Node (노드)

State를 입력받아 **처리한 뒤 업데이트된 State를 반환**하는 함수이다.

```python
def my_node(state: MyState) -> MyState:
    # state에서 데이터를 읽어 처리하고
    result = some_processing(state['question'])
    # 업데이트할 필드만 반환하면 된다 (전체를 반환할 필요 없음)
    return {'answer': result}
```

- 각 노드는 **하나의 역할**만 담당한다 (검색 노드, 생성 노드 등)
- 비유: **작업자** — 칠판에서 정보를 읽고, 자기 작업 결과를 칠판에 적는다

#### ③ Edge (엣지)

노드 간의 **실행 순서(연결)**를 정의한다.

```python
workflow.add_edge(START, 'node_a')    # 시작 → node_a
workflow.add_edge('node_a', 'node_b') # node_a → node_b
workflow.add_edge('node_b', END)      # node_b → 끝
```

- `START`와 `END`는 LangGraph가 제공하는 특수 노드 (시작점, 종료점)
- 비유: **컨베이어 벨트** — 작업 순서를 결정한다

<br>

> **⭐ 세 요소를 하나의 비유로**
>
> ```
>    State  =  칠판       모두가 함께 보는 공유 메모장
>    Node   =  작업자     칠판을 읽고, 자기 일을 하고, 결과를 칠판에 적는다
>    Edge   =  순서표     누가 먼저 하고 누가 나중에 할지 정한다
> ```
>
> 이 비유를 기억하면 아래 코드가 훨씬 쉽게 읽힌다.

> **📌 왜 노드가 State 전체가 아니라 '일부'만 반환해도 되는가?**
>
> ```python
> def my_node(state): 
>     return {'answer': result}   # answer만 반환. question은 안 건드림
> ```
>
> LangGraph가 **반환된 필드만 골라서 State에 덮어써 주기** 때문이다.
> 칠판 전체를 다시 쓸 필요 없이, **내가 담당한 칸만 고쳐 쓰면** 된다.

### 6-7. LangGraph 기본 문법 — 4단계 코드 패턴

LangGraph로 워크플로우를 만드는 코드는 항상 **동일한 4단계 패턴**을 따른다.

```python
# 1단계: State 정의 (TypedDict)
class MyState(TypedDict):
    ...

# 2단계: Node 함수 정의
def node_a(state: MyState) -> MyState:
    ...

# 3단계: Graph 구성 (노드 추가 + 엣지 연결)
workflow = StateGraph(MyState)
workflow.add_node('node_a', node_a)
workflow.add_edge(START, 'node_a')
workflow.add_edge('node_a', END)

# 4단계: 컴파일 + 실행
graph = workflow.compile()
result = graph.invoke({'question': '...'})
```

> **💡 이 4단계는 어떤 워크플로우를 만들든 똑같다.**
> 바뀌는 것은 State의 필드와 Node의 개수·내용뿐이다.
> 1-2 챕터의 "학습 루프 5줄"처럼, **골격을 외워두면 응용이 쉬워진다.**

아래에서 이 패턴을 간단한 예제로 먼저 익힌 뒤, RAG 파이프라인에 적용한다.


### 6-8. LangGraph 기초 실습 — Hello World

RAG 파이프라인을 만들기 전에, 가장 간단한 예제로 LangGraph의 동작 방식을 이해하자.

목표: 사용자의 이름을 받아 **인사말을 생성**하는 2단계 워크플로우

```
START → [format 노드] → [greet 노드] → END
         이름을 정리      인사말 생성
```


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ========== 1단계: State 정의 ==========
# 이 그래프에서 오가는 데이터의 구조를 정의한다.
class GreetState(TypedDict):
    name: str       # 사용자 이름 (입력)
    formatted: str  # 정리된 이름 (중간 결과)
    greeting: str   # 최종 인사말 (출력)

# ========== 2단계: Node 함수 정의 ==========
# 각 노드는 state를 받아서, 업데이트할 필드만 딕셔너리로 반환한다.

def format_name(state: GreetState) -> GreetState:
    """이름을 정리하는 노드: 공백 제거 + 첫 글자 대문자"""
    name = state['name'].strip()
    return {'formatted': name}  # 'formatted' 필드만 업데이트

def greet(state: GreetState) -> GreetState:
    """인사말을 생성하는 노드"""
    greeting = f'안녕하세요, {state["formatted"]}님! LangGraph에 오신 걸 환영합니다.'
    return {'greeting': greeting}  # 'greeting' 필드만 업데이트

# ========== 3단계: Graph 구성 ==========
workflow = StateGraph(GreetState)

# 노드 추가: add_node('노드이름', 함수)
workflow.add_node('format', format_name)
workflow.add_node('greet', greet)

# 엣지 연결: add_edge(출발, 도착)
workflow.add_edge(START, 'format')   # 시작 → format
workflow.add_edge('format', 'greet') # format → greet
workflow.add_edge('greet', END)      # greet → 끝

# ========== 4단계: 컴파일 + 실행 ==========
hello_graph = workflow.compile()

# invoke: 초기 State를 넣으면, 모든 노드를 순서대로 실행하고 최종 State를 반환
result = hello_graph.invoke({'name': '  김싸피  '})

print(f'입력: "{result["name"]}"')
print(f'정리: "{result["formatted"]}"')
print(f'결과: "{result["greeting"]}"')
print('\n→ State가 노드를 거치며 단계적으로 업데이트되는 것을 확인!')


### 6-9. [참고] 조건부 엣지 (Conditional Edge)

LangGraph의 강력한 기능 중 하나는 **조건에 따라 다른 노드로 분기**할 수 있다는 것이다.

```python
# 조건부 분기 예시
def route_decision(state: MyState) -> str:
    """state 내용에 따라 다음 노드를 결정하는 함수"""
    if state['context'] == '':
        return 'fallback'    # 검색 결과 없음 → fallback 노드로
    else:
        return 'generate'    # 검색 결과 있음 → generate 노드로

workflow.add_conditional_edges(
    'retrieve',              # 출발 노드
    route_decision,          # 분기 판단 함수
    {'generate': 'generate', 'fallback': 'fallback'}  # 반환값 → 노드 매핑
)
```

```
              ┌─ context 있음 → [generate] → END
START → [retrieve] ─┤
              └─ context 없음 → [fallback] → END
```

> **💡 이 실습에서는 기본 엣지만 사용하지만,**
> 실무에서는 조건부 엣지로 "검색 실패 시 재검색", "답변 품질 낮으면 재생성" 등의
> 고급 워크플로우를 구현할 수 있다.

### 6-10. LangGraph로 RAG 파이프라인 구현

이제 Hello World에서 익힌 4단계 패턴을 **RAG 파이프라인에 그대로 적용**한다.

<!-- 🖼️ 이미지 위치 B: RAG LangGraph 파이프라인 다이어그램 -->

```
START → [retrieve 노드] → [generate 노드] → END
         질문으로 검색       검색 결과로 답변 생성
```

| Hello World | RAG 파이프라인 |
|:---:|:---:|
| `GreetState` | `RAGState` |
| `format_name` 노드 | `retrieve` 노드 (Vector Store 검색) |
| `greet` 노드 | `generate` 노드 (LLM 답변 생성) |


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ========== 1. State 정의 ==========
# 그래프 전체에서 공유되는 데이터 구조 ("칠판")
# 각 노드는 이 State를 읽고 업데이트한다.
class RAGState(TypedDict):
    question: str   # 사용자 질문
    context: str    # 검색된 문서 내용
    answer: str     # 생성된 답변

# ========== 2. retrieve 노드 ==========
# 역할: 질문으로 Vector Store에서 관련 문서를 검색한다.
def retrieve(state: RAGState) -> RAGState:
    """Vector Store에서 관련 문서를 검색하는 노드"""
    docs = retriever_chunked.invoke(state['question'])
    context = '\n\n'.join([doc.page_content for doc in docs])
    return {'context': context}

# ========== 3. generate 노드 ==========
# 역할: 검색된 문서를 참고하여 LLM이 답변을 생성한다.
def generate(state: RAGState) -> RAGState:
    """검색된 문서를 기반으로 답변을 생성하는 노드"""
    rag_prompt = ChatPromptTemplate.from_messages([
        ('system', '''당신은 온라인 서점 Yes24의 고객 서비스 상담원입니다.
다음 검색된 자료를 참고하여 고객의 질문에 정확하고 친절하게 답변해주세요.

[검색된 자료]
{context}

답변 규칙:
1. 검색된 자료를 기반으로 답변하세요
2. 자료에 없는 내용은 추측하지 마세요
3. 존댓말을 사용하세요'''),
        ('human', '{question}')
    ])
    messages = rag_prompt.format_messages(
        context=state['context'], question=state['question']
    )
    response = llm.invoke(messages)
    return {'answer': response.content}

# ========== 4. StateGraph 구성 ==========
workflow = StateGraph(RAGState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('generate', generate)
workflow.add_edge(START, 'retrieve')
workflow.add_edge('retrieve', 'generate')
workflow.add_edge('generate', END)

rag_graph = workflow.compile()
print('LangGraph RAG 파이프라인 구성 완료!')


In [ ]:
# ========== RAG 파이프라인 실행 ==========
test_questions = [
    'Yes24에서 총알배송이 뭔가요? 서울에서 당일배송 마감 시간은?',
    '배송이 늦으면 보상받을 수 있나요?',
    'YES포인트는 유효기간이 있나요?',
    '신규 회원 혜택은 무엇인가요?',
]

for q in test_questions:
    result = rag_graph.invoke({'question': q})
    print(f'질문: {q}')
    print(f'답변: {result["answer"][1]['text']}')
    print('=' * 60)


In [ ]:
# ========== 최종 비교: LLM만 vs RAG ==========
comparison_q = 'Yes24 배송지연 보상제도는 어떻게 되나요? 보상 금액은?'

# LLM만
llm_only = llm.invoke([HumanMessage(content=comparison_q)])
# RAG
rag_result = rag_graph.invoke({'question': comparison_q})

print('=' * 60)
print('최종 비교: LLM만 vs RAG')
print('=' * 60)
print(f'\n질문: {comparison_q}')
print(f'\n--- LLM만 (환각 위험) ---')
print(llm_only.content[1]['text'][:300])
print(f'\n--- RAG (자료 기반) ---')
print(rag_result['answer'][1]['text'][:300])
print(f'\n정답: 주문 건당 YES포인트 2,000원 (실제 PDF 기준)')


---

## 정리

### 오늘 배운 전체 흐름

```
① LLM만 사용 → 환각 발생 (챕터 1)
② RAG = "답변 전에 자료를 먼저 찾아 읽어라" (챕터 2)
③ 키워드 검색 → 동의어 인식 불가 (챕터 3)
④ 임베딩 + Vector Store → 의미 기반 검색 (챕터 4)
⑤ 청킹으로 검색 정밀도 향상 (챕터 5)
⑥ LangGraph로 전체 파이프라인 자동화 (챕터 6)
```

### 핵심 개념 요약

| 개념 | 한 줄 정리 |
|------|----------|
| **RAG** | "답변 전에 자료를 먼저 찾아 읽어라" — 오픈북 시험 |
| **임베딩** | 텍스트를 숫자 벡터로 변환 → 의미가 비슷하면 벡터도 비슷 |
| **Vector Store** | 임베딩 벡터를 저장하고 유사도 검색하는 DB |
| **청킹** | 긴 문서를 적절한 크기로 잘라 검색 정밀도 향상 |
| **Retriever** | Vector Store에서 질문과 유사한 문서를 찾아 반환 |
| **LangChain** | LLM 앱을 만드는 **부품 상자** (Loader·Splitter·Embedding·VectorStore) |
| **LangGraph** | State + Node + Edge로 **워크플로우를 조립**하는 프레임워크 |

### LangChain과 LangGraph의 관계

```
   LangChain   =  부품 상자      "무엇으로 만들 것인가"
   LangGraph   =  조립 설계도    "어떤 순서로 엮을 것인가"

   이 노트북 : LangChain 부품을 쓰면서  ->  LangGraph로 조립했다
```

| | LangChain (Chain) | LangGraph (Graph) |
|---|---|---|
| 흐름 | 한 방향으로 쭉 | **분기·반복·되돌아가기** |
| 적합한 작업 | 정해진 순서의 단순 작업 | 에이전트, 복잡한 워크플로우 |

<br>

> 💡 LangChain의 문법(LCEL 등)은 **자기주도 학습 자료**에서 별도로 다룬다.

### RAG 파이프라인 전체 구성도

```
[준비 단계]
PDF → PyMuPDF4LLMLoader → 청킹(300자) → 임베딩 → ChromaDB 저장

[실행 단계 — LangGraph]
질문 → retrieve 노드(검색) → generate 노드(생성) → 정확한 답변
```

### 이번 실습의 환경 설정 요약

| 항목 | 설정값 |
|---|---|
| API 제공처 | **SSAFY GMS** (`https://gms.ssafy.io/gmsapi/api.openai.com/v1/`) |
| 인증 | `.env`의 `GMS_KEY` |
| LLM | `gpt-5-nano` + **Responses API** (`use_responses_api=True`) |
| 추론 강도 | `reasoning_effort='low'` (temperature 대신) |
| 임베딩 | `text-embedding-3-small` (1536차원) |

<br>

> **⚠️ GPT-5 계열에서 주의할 점**
>
> `temperature`와 `top_p`는 **기본값만 허용**된다. 다른 값을 지정하면 오류가 발생한다.
> 응답의 성격을 조절하려면 `reasoning_effort`(low / medium / high)를 사용한다.
> (2-2 챕터에서 다룬 내용과 동일하다)

### 🔬 직접 해볼 실험

| # | 실험 | 바꿀 것 | 관찰할 것 |
|:--:|---|---|---|
| 1 | 추론 강도 | `reasoning_effort`를 `low` -> `medium` | 답변 품질과 응답 시간의 변화 |
| 2 | 검색 개수 | `search_kwargs={'k': 3}` -> `k: 1` 또는 `5` | 자료가 적거나 많을 때 답변이 어떻게 달라지는가 |
| 3 | 청크 크기 | `chunk_size`를 300 -> 100 또는 1000 | 검색 정밀도와 문맥 보존의 균형 |
| 4 | **없는 정보 질문** | 자료에 없는 내용을 물어보기 | "자료에 없습니다"라고 제대로 답하는가 ⭐ |
| 5 | 검색 없이 생성 | `retrieve` 노드를 건너뛰기 | 검색이 빠지면 답변이 어떻게 무너지는가 |
| 6 | 조건부 엣지 | 6-9절 코드로 fallback 노드 추가 | 검색 실패 시 흐름이 갈라지는가 |

<br>

> **⭐ 4번 실험을 꼭 해보세요**
>
> 예: `"Yes24 해외배송은 며칠 걸리나요?"` (자료에 없는 내용)
>
> 프롬프트에 **"자료에 없는 내용은 추측하지 마세요"** 라고 적어두었기 때문에
> **"제공된 자료에 없습니다"** 라고 답해야 정상이다.
>
> 만약 그럴듯한 답을 지어낸다면? **RAG를 써도 환각이 완전히 사라지지는 않는다**는 증거다.
> 이것이 1장에서 말한 RAG의 한계이며, 실무에서 **답변 검증 단계**를 따로 두는 이유다.

### 🐛 자주 만나는 문제

| 증상 | 원인 | 해결 |
|---|---|---|
| `PDF 파일이 없습니다` | `data/` 폴더 위치가 다름 | 출력된 `resolve()` 경로 확인 |
| 검색이 엉뚱한 문서를 찾음 | 청크가 너무 크거나 작음 | `chunk_size` 조정 후 Vector Store 재생성 |
| Vector Store가 갱신 안 됨 | 같은 `collection_name` 재사용 | 이름을 바꾸거나 기존 컬렉션 삭제 |
| 답변이 자료와 다름 | 검색 단계에서 이미 실패 | `retriever.invoke(질문)`로 **검색 결과부터** 확인 |
| `Unsupported parameter` | `temperature` 지정 | GPT-5 계열은 `reasoning_effort` 사용 |

<br>

> **💡 RAG 디버깅의 원칙: 검색부터 확인하라**
>
> 답변이 이상하면 LLM을 탓하기 쉽지만, **대부분 원인은 검색 단계**에 있다.
> `retriever.invoke(질문)`을 직접 실행해서 **어떤 문서가 걸렸는지** 먼저 보자.
> 엉뚱한 문서가 걸렸다면 LLM이 아무리 좋아도 정답을 낼 수 없다.

### 실습 안내

이제 `실습_4-1_RAG_기반_Customer_Service_AI_에이전트_개발.ipynb`를 열고,
오늘 배운 개념을 TODO 코드로 직접 구현해 보자.

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.
